In [1]:
# Cell 1: imports and basic settings

import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\PRAGA\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\PRAGA\AppData\Roaming\Python\Python312\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\PRAGA\AppData\Roaming\Python\Python312\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io_lo

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\PRAGA\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\PRAGA\AppData\Roaming\Python\Python312\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\PRAGA\AppData\Roaming\Python\Python312\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io_lo

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



In [2]:
# Cell 2: load CSV and basic look

df = pd.read_csv("project_sales_data.csv")

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'project_sales_data.csv'

In [ ]:
# Cell 3: clean column names and replace bad tokens

df.columns = [col.strip().replace(" ", "_") for col in df.columns]

bad_values = ["#VALUE!", "VALUE!", "", " ", "NA", "N/A", "null", "NULL", "None"]
df = df.replace(bad_values, np.nan)

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip()

df = df.replace("nan", np.nan)

df.head()

,Created,Product_ID,Source,Mobile,EMAIL,Sales_Agent,Location,Delivery_Mode,Status
0,14-11-2018 10:05,NaN,Website,984XXXXXXX,aXXXXXXX@gmail.com,Sales-Agent-11,NaN,Mode-5,Open
1,14-11-2018 09:22,NaN,Website,XXXXXXX,NaN,Sales-Agent-10,NaN,Mode-5,Open
2,14-11-2018 09:21,NaN,Website,XXXXXXX,dXXXXXXX@yahoo.com,Sales-Agent-10,NaN,Mode-5,Open
3,14-11-2018 08:46,NaN,Website,XXXXXXX,wXXXXXXX@gmail.com,Sales-Agent-10,NaN,Mode-5,Open
4,14-11-2018 07:34,NaN,Website,XXXXXXX,cXXXXXXX@gmail.com,Sales-Agent-10,NaN,Mode-5,Open


In [ ]:
# Cell 4: parse Created and add time-based features

df["Created"] = pd.to_datetime(df["Created"], format="%d-%m-%Y %H:%M", errors="coerce")

df["Created_Day"] = df["Created"].dt.day
df["Created_Month"] = df["Created"].dt.month
df["Created_Year"] = df["Created"].dt.year
df["Created_Hour"] = df["Created"].dt.hour
df["Created_DayOfWeek"] = df["Created"].dt.day_name()

df[["Created", "Created_Day", "Created_Month", "Created_Year", "Created_Hour", "Created_DayOfWeek"]].head()

,Created,Created_Day,Created_Month,Created_Year,Created_Hour,Created_DayOfWeek
0,2018-11-14 10:05:00,14,11,2018,10,Wednesday
1,2018-11-14 09:22:00,14,11,2018,9,Wednesday
2,2018-11-14 09:21:00,14,11,2018,9,Wednesday
3,2018-11-14 08:46:00,14,11,2018,8,Wednesday
4,2018-11-14 07:34:00,14,11,2018,7,Wednesday


In [ ]:
# Cell 5: normalize Status

df["Status"] = df["Status"].str.strip().str.upper()

print(df["Status"].value_counts(dropna=False))

Status
JUNK LEAD               1536
NOT RESPONDING          1129
CONVERTED                852
JUST ENQUIRY             760
POTENTIAL                708
LONG TERM                646
IN PROGRESS POSITIVE     643
IN PROGRESS NEGATIVE     626
LOST                     440
OPEN                      82
Name: count, dtype: int64


In [ ]:
# Cell 6: map multi-class Status to binary Lead_Category

high_status = ["CONVERTED", "POTENTIAL", "LONG TERM", "IN PROGRESS POSITIVE"]
low_status  = ["OPEN", "LOST", "JUST ENQUIRY", "NOT RESPONDING", "JUNK LEAD", "IN PROGRESS NEGATIVE"]

def map_lead_category(status):
    if status in high_status:
        return "High"
    elif status in low_status:
        return "Low"
    else:
        return np.nan

df["Lead_Category"] = df["Status"].apply(map_lead_category)

df["Lead_Category"].value_counts(dropna=False)

,count
Lead_Category,
Low,4573
High,2849


In [ ]:
# Cell 7: basic engineered features and dropping unmapped rows

# Drop rows where we don’t have a High/Low label
df = df.dropna(subset=["Lead_Category"])

# Product_ID as string category
df["Product_ID"] = df["Product_ID"].astype(str)

# Presence flags for contact info
df["Has_Mobile"] = df["Mobile"].notna().astype(int)
df["Has_Email"] = df["EMAIL"].notna().astype(int)

def email_quality(x):
    if pd.isna(x):
        return "Missing"
    x = str(x).strip().lower()
    if x in ["#value!", "value!", "nan"]:
        return "Missing"
    if "@" in x and "." in x:
        return "Looks_Valid"
    return "Suspicious"

df["Email_Quality"] = df["EMAIL"].apply(email_quality)

df[["Product_ID", "Source", "Sales_Agent", "Location", "Delivery_Mode",
    "Has_Mobile", "Has_Email", "Email_Quality", "Lead_Category"]].head()

,Product_ID,Source,Sales_Agent,Location,Delivery_Mode,Has_Mobile,Has_Email,Email_Quality,Lead_Category
0,nan,Website,Sales-Agent-11,NaN,Mode-5,1,1,Looks_Valid,Low
1,nan,Website,Sales-Agent-10,NaN,Mode-5,1,0,Missing,Low
2,nan,Website,Sales-Agent-10,NaN,Mode-5,1,1,Looks_Valid,Low
3,nan,Website,Sales-Agent-10,NaN,Mode-5,1,1,Looks_Valid,Low
4,nan,Website,Sales-Agent-10,NaN,Mode-5,1,1,Looks_Valid,Low


In [ ]:
# Cell 8: select features and split

features = [
    "Product_ID",
    "Source",
    "Sales_Agent",
    "Location",
    "Delivery_Mode",
    "Created_Day",
    "Created_Month",
    "Created_Year",
    "Created_Hour",
    "Created_DayOfWeek",
    "Has_Mobile",
    "Has_Email",
    "Email_Quality"
]

target = "Lead_Category"

X = df[features]
y = df[target].map({"Low": 0, "High": 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train.head()

,Product_ID,Source,Sales_Agent,Location,Delivery_Mode,Created_Day,Created_Month,Created_Year,Created_Hour,Created_DayOfWeek,Has_Mobile,Has_Email,Email_Quality
1201,19.0,Live Chat -PPC,Sales-Agent-3,Hyderabad,Mode-3,6,10,2018,11,Saturday,1,1,Looks_Valid
2157,5.0,Live Chat-Direct,Sales-Agent-9,Bangalore,Mode-1,8,9,2018,13,Saturday,1,1,Looks_Valid
4356,20.0,E-mail Campaign,Sales-Agent-12,Chennai,Mode-3,11,7,2018,18,Wednesday,1,1,Suspicious
906,15.0,Call,Sales-Agent-9,Other Locations,Mode-5,15,10,2018,12,Monday,1,0,Missing
2264,27.0,Live Chat-Blog,Sales-Agent-11,Other Locations,Mode-3,5,9,2018,18,Wednesday,1,1,Looks_Valid


In [ ]:
# Cell 9: preprocessing (numeric + categorical)

numeric_features = [
    "Created_Day",
    "Created_Month",
    "Created_Year",
    "Created_Hour",
    "Has_Mobile",
    "Has_Email"
]

categorical_features = [
    "Product_ID",
    "Source",
    "Sales_Agent",
    "Location",
    "Delivery_Mode",
    "Created_DayOfWeek",
    "Email_Quality"
]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [ ]:
# Cell 10: define models to compare

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        random_state=42,
        class_weight="balanced"
    ),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

In [ ]:
# Cell 11: train each model and evaluate

results = []
best_model_name = None
best_pipeline = None
best_f1 = -1.0

for name, model in models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    if hasattr(pipe.named_steps["model"], "predict_proba"):
        y_prob = pipe.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_prob)
    else:
        y_prob = None
        auc = np.nan

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1,
        "ROC_AUC": auc
    })

    print(f"\n==== {name} ====")
    print("Accuracy :", round(acc, 4))
    print("Precision:", round(prec, 4))
    print("Recall   :", round(rec, 4))
    print("F1       :", round(f1, 4))
    print("ROC AUC  :", round(auc, 4) if not np.isnan(auc) else "N/A")

    print("\nConfusion matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification report:")
    print(classification_report(y_test, y_pred, target_names=["Low", "High"]))

    if f1 > best_f1:
        best_f1 = f1
        best_model_name = name
        best_pipeline = pipe


==== Logistic Regression ====
Accuracy : 0.7185
Precision: 0.6064
Recall   : 0.7596
F1       : 0.6745
ROC AUC  : 0.8177

Confusion matrix:
[[634 281]
 [137 433]]

Classification report:
              precision    recall  f1-score   support

         Low       0.82      0.69      0.75       915
        High       0.61      0.76      0.67       570

    accuracy                           0.72      1485
   macro avg       0.71      0.73      0.71      1485
weighted avg       0.74      0.72      0.72      1485


==== Random Forest ====
Accuracy : 0.7192
Precision: 0.6584
Recall   : 0.5579
F1       : 0.604
ROC AUC  : 0.7954

Confusion matrix:
[[750 165]
 [252 318]]

Classification report:
              precision    recall  f1-score   support

         Low       0.75      0.82      0.78       915
        High       0.66      0.56      0.60       570

    accuracy                           0.72      1485
   macro avg       0.70      0.69      0.69      1485
weighted avg       0.71      0.72 

In [ ]:
# Cell 12: model comparison summary and saving artifacts

results_df = pd.DataFrame(results).sort_values(by="F1", ascending=False)
results_df

,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Logistic Regression,0.718519,0.606443,0.759649,0.674455,0.817722
2,Gradient Boosting,0.752189,0.721491,0.577193,0.641326,0.822412
1,Random Forest,0.719192,0.658385,0.557895,0.603989,0.795357


In [ ]:
print("Best model:", best_model_name)
print("Best F1 score:", round(best_f1, 4))

# Save cleaned data and results
df.to_csv("project_sales_cleaned_with_target.csv", index=False)
results_df.to_csv("model_comparison_results.csv", index=False)

# Save best model pipeline
joblib.dump(best_pipeline, "best_lead_category_model.pkl")

print("Saved:")
print("- project_sales_cleaned_with_target.csv")
print("- model_comparison_results.csv")
print("- best_lead_category_model.pkl")

Best model: Logistic Regression
Best F1 score: 0.6745
Saved:
- project_sales_cleaned_with_target.csv
- model_comparison_results.csv
- best_lead_category_model.pkl


In [ ]:
# Cell 13: feature importance (only for Random Forest best model)

if best_model_name == "Random Forest":
    ohe = best_pipeline.named_steps["preprocessor"].named_transformers_["cat"].named_steps["onehot"]
    cat_feature_names = ohe.get_feature_names_out(categorical_features)
    all_feature_names = numeric_features + list(cat_feature_names)

    importances = best_pipeline.named_steps["model"].feature_importances_
    feat_imp_df = pd.DataFrame({
        "Feature": all_feature_names,
        "Importance": importances
    }).sort_values(by="Importance", ascending=False)

    feat_imp_df.head(20)
    feat_imp_df.to_csv("feature_importance.csv", index=False)
    print("Saved feature_importance.csv")

In [ ]:
# Cell 14: example prediction for one new lead

sample = pd.DataFrame([{
    "Product_ID": "18",
    "Source": "Website",
    "Sales_Agent": "Sales-Agent-4",
    "Location": "Bangalore",
    "Delivery_Mode": "Mode-1",
    "Created_Day": 3,
    "Created_Month": 10,
    "Created_Year": 2018,
    "Created_Hour": 14,
    "Created_DayOfWeek": "Wednesday",
    "Has_Mobile": 1,
    "Has_Email": 1,
    "Email_Quality": "Looks_Valid"
}])

pred = best_pipeline.predict(sample)[0]
label = "High" if pred == 1 else "Low"
print("Predicted lead category:", label)

if hasattr(best_pipeline.named_steps["model"], "predict_proba"):
    print("Probabilities [Low, High]:", best_pipeline.predict_proba(sample)[0])

Predicted lead category: High
Probabilities [Low, High]: [0.30959262 0.69040738]
